# Coursework 1

This notebook is intended to be used as a starting point for your experiments. The instructions can be found in the MLP2025_26_CW1_Spec.pdf (see Learn,  Assignment Submission, Coursework 1). The methods provided here are just helper functions. If you want more complex graphs such as side by side comparisons of different experiments you should learn more about matplotlib and implement them. Before each experiment remember to re-initialize neural network weights and reset the data providers so you get a properly initialized experiment. For each experiment try to keep most hyperparameters the same except the one under investigation so you can understand what the effects of each are.

## Training Boilerplate

Use the below code as a boilerplate to start your experiments. You can add more cells or change the code as you see fit.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import logging
# import sys
# sys.path.append('/path/to/mlpractical')

from mlp.data_providers import MNISTDataProvider, EMNISTDataProvider
from mlp.layers import AffineLayer, SoftmaxLayer, SigmoidLayer, ReluLayer, CustomActivationLayer, DropoutLayer
from mlp.errors import CrossEntropySoftmaxError
from mlp.models import MultipleLayerModel
from mlp.initialisers import ConstantInit, GlorotUniformInit
from mlp.learning_rules import AdamLearningRule
from mlp.optimisers import Optimiser
from mlp.penalties import L1Penalty, L2Penalty, L1L2MixPenalty

In [2]:
%matplotlib inline
plt.style.use('ggplot')

def train_model_and_plot_stats(
        model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True):
    
    # As well as monitoring the error over training also monitor classification
    # accuracy i.e. proportion of most-probable predicted classes being equal to targets
    data_monitors={'acc': lambda y, t: (y.argmax(-1) == t.argmax(-1)).mean()}

    # Use the created objects to initialise a new Optimiser instance.
    optimiser = Optimiser(
        model, error, learning_rule, train_data, valid_data, data_monitors, notebook=notebook)

    # Run the optimiser for num_epochs epochs (full passes through the training set)
    # printing statistics every epoch.
    stats, keys, run_time = optimiser.train(num_epochs=num_epochs, stats_interval=stats_interval)

    # Plot the change in the validation and training set error over training.
    fig_1 = plt.figure(figsize=(8, 4))
    ax_1 = fig_1.add_subplot(111)
    for k in ['error(train)', 'error(valid)']:
        ax_1.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]], label=k)
    ax_1.legend(loc=0)
    ax_1.set_xlabel('Epoch number')
    ax_1.set_ylabel('Error')

    # Plot the change in the validation and training set accuracy over training.
    fig_2 = plt.figure(figsize=(8, 4))
    ax_2 = fig_2.add_subplot(111)
    for k in ['acc(train)', 'acc(valid)']:
        ax_2.plot(np.arange(1, stats.shape[0]) * stats_interval, 
                  stats[1:, keys[k]], label=k)
    ax_2.legend(loc=0)
    ax_2.set_xlabel('Epoch number')
    ax_2.set_xlabel('Accuracy')

    grad_plot, grad_ax = optimiser.plot_grad_flow()

    return stats, keys, run_time, fig_1, ax_1, fig_2, ax_2, grad_plot, grad_ax

In [3]:
# The below code will set up the data providers, random number
# generator and logger objects needed for training runs. As
# loading the data from file take a little while you generally
# will probably not want to reload the data providers on
# every training run. If you wish to reset their state you
# should instead use the .reset() method of the data providers.

# Seed a random number generator
seed = 111020
rng = np.random.RandomState(seed)
batch_size = 100
# Set up a logger object to print info about the training run to stdout
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers = [logging.StreamHandler()]

# Create data provider objects for the MNIST data set
train_data = EMNISTDataProvider('train', batch_size=batch_size, rng=rng)
valid_data = EMNISTDataProvider('valid', batch_size=batch_size, rng=rng)

KeysView(NpzFile '/Users/alicegraham/Documents/GitHub/mlpractical/data/emnist-train.npz' with keys: inputs, targets)
KeysView(NpzFile '/Users/alicegraham/Documents/GitHub/mlpractical/data/emnist-valid.npz' with keys: inputs, targets)


# L1L2Mix Penalty #

In [15]:
# L1 L2 Mix PENALTY

# Setup hyperparameters
learning_rate = 0.0001
num_epochs = 10
stats_interval = 1
input_dim, output_dim, hidden_dim = 784, 47, 128

weights_init = GlorotUniformInit(rng=rng)
biases_init = ConstantInit(0.)

def compute_adaptive_weights(w_hat, gamma=1.5, eps=1e-8):
    """Compute adaptive weights a_i = 1 / (|w_hat_i|^γ + eps)"""
    return 1.0 / (np.abs(w_hat) ** gamma + eps)

lam = 0.001
alpha = 0.5  # L1/L2 mixing
gamma = 1


# -----------------------------
# Initial weight estimates
# -----------------------------
# First hidden layer: OLS initialization
# X_train: shape (n_samples, 784)
# For hidden layer, create dummy target
X_batch, y_batch = train_data.next()  # returns (batch_size, 784), (batch_size, 47)
n_samples = X_batch.shape[0]

# dummy target for hidden layer initialization
y_dummy = np.random.randn(n_samples, hidden_dim)

XtX_inv = np.linalg.pinv(X_batch.T @ X_batch)
w_hat_1 = XtX_inv @ (X_batch.T @ y_dummy)
adaptive_weights_1 = compute_adaptive_weights(w_hat_1, gamma=gamma)

# Deeper layers: random init
w_hat_2 = np.random.randn(hidden_dim, hidden_dim) * 0.01
w_hat_3 = np.random.randn(hidden_dim, hidden_dim) * 0.01
w_hat_out = np.random.randn(hidden_dim, output_dim) * 0.01

adaptive_weights_2 = compute_adaptive_weights(w_hat_2, gamma=gamma)
adaptive_weights_3 = compute_adaptive_weights(w_hat_3, gamma=gamma)
adaptive_weights_out = compute_adaptive_weights(w_hat_out, gamma=gamma)

# -----------------------------
# Penalty objects
# -----------------------------
weights_penalty_1 = L1L2MixPenalty(lam, alpha, adaptive_weights_1.T)
weights_penalty_2 = L1L2MixPenalty(lam, alpha, adaptive_weights_2.T)
weights_penalty_3 = L1L2MixPenalty(lam, alpha, adaptive_weights_3.T)
weights_penalty_out = L1L2MixPenalty(lam, alpha, adaptive_weights_out.T)

# For biases, just use ones
# Hidden layers biases penalty
biases_penalty_hidden = L1L2MixPenalty(lam, alpha, np.ones(hidden_dim))

# Output layer biases penalty
biases_penalty_out = L1L2MixPenalty(lam, alpha, np.ones(output_dim))


# Create model with THREE hidden layers
l2_model = MultipleLayerModel([
    AffineLayer(input_dim, hidden_dim, weights_init, biases_init, weights_penalty=weights_penalty_1, biases_penalty=biases_penalty_hidden), # first hidden layer
    ReluLayer(),

    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty=weights_penalty_2, biases_penalty=biases_penalty_hidden), # second hidden layer
    ReluLayer(),

    AffineLayer(hidden_dim, hidden_dim, weights_init, biases_init, weights_penalty=weights_penalty_3, biases_penalty=biases_penalty_hidden), # third hidden layer
    ReluLayer(),

    AffineLayer(hidden_dim, output_dim, weights_init, biases_init, weights_penalty=weights_penalty_out, biases_penalty=biases_penalty_out) # output layer
])

error = CrossEntropySoftmaxError()
# Use a Adam learning rule
learning_rule = AdamLearningRule(learning_rate=learning_rate)

# Remember to use notebook=False when you write a script to be run in a terminal
l2_stats, l2_keys, l2_run_time, l2_fig_1, l2_ax_1, l2_fig_2, l2_ax_2, l2_grad_plot, l2_grad_ax = train_model_and_plot_stats(
    l2_model, error, learning_rule, train_data, valid_data, num_epochs, stats_interval, notebook=True)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 1: 14.5s to complete
    error(train)=3.85e+00, acc(train)=1.06e-01, error(valid)=3.85e+00, acc(valid)=1.02e-01


  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 2: 14.0s to complete
    error(train)=3.85e+00, acc(train)=2.17e-02, error(valid)=3.85e+00, acc(valid)=2.01e-02


...

Epoch 6: 13.4s to complete
    error(train)=3.85e+00, acc(train)=2.17e-02, error(valid)=3.85e+00, acc(valid)=2.01e-02


  0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 7: 16.2s to complete
    error(train)=3.85e+00, acc(train)=2.17e-02, error(valid)=3.85e+00, acc(valid)=2.01e-02


  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 